# 10 · Gradient geometry — worked solutions

These solutions are most useful after attempting the chapter exercises. Each
section states the reasoning before the calculation; numerical values depend on
the compact, fixed-seed setup below.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=30.0,
    length=24_000.0, width=12_000.0, n_length=4, n_width=3,
)
i = np.arange(fault.n_patches) % 4
j = np.arange(fault.n_patches) // 4
true_slip = np.exp(-((i - 1.5) / 1.2) ** 2 - ((j - 1.0) / 0.9) ** 2)
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.18, -117.82, 6), np.linspace(33.90, 34.12, 5)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.004
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="solution_gnss",
)
geodef.backend.set_backend("jax")


## 1–3. Dimension, timing, and scaling

Add free parameters gradually and record both compile and repeated execution
time. Scale meter-valued parameters so optimizer steps are commensurate with
degree-valued parameters.

## 4. Finite-difference validation

A centered difference should agree with the autodiff gradient over a range of
step sizes before cancellation dominates. Agreement validates the implemented
derivative, not the geological model.


In [ ]:
theta = np.array([0.0, 0.0, 9_000.0, 90.0, 30.0, 24_000.0, 12_000.0])
search = geodef.invert.geometry_search(
    theta, gnss, ref_lat=34.0, ref_lon=-118.0,
    free=["dip"], bounds={"dip": (15.0, 55.0)},
    n_length=4, n_width=3, components="dip",
    regularization="laplacian", regularization_strength=1.0,
)
print("optimized dip:", search.theta[4])
print("local dip sigma:", np.sqrt(search.theta_cov[0, 0]))
geodef.backend.set_backend("numpy")


## Interpretation and alternatives

If the scan from Chapter 09 shows more than one basin, the local sigma printed here must not be reported as total geometry uncertainty.
